In [1]:
import yfinance as yf
import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [15]:
stocks = ["AMZN", "MSFT", "GOOG"]  # List of stock tickers
#start = dt.datetime.today() - dt.timedelta(360) # 30 days ago
#end = dt.datetime.today()

ohlcv_data = {}  # Dictionary to hold OHLCV data. Key-Value pair: Ticker-DataFrame

# get closing prices for each stock
for ticker in stocks:
    temp = yf.download(ticker, period="1y", interval="1d")
    temp.dropna(how="any", inplace=True) # type: ignore
    ohlcv_data[ticker] = temp


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [16]:
# this ratio is the avg return earned in excess of risk free per unit of volatility
# sharpe ratio > 1 is good, > 2 even better and > 3 is excellent
# sortino ratio takes into account stdev of only negative returns

# sharpe ratio = (R(p) - R(f)) / stdev(p)
# sortino ratio = (R(p) - R(f)) / stdev(p) but stdev is only of negative returns

In [17]:
def cagr(DF):
    df = DF.copy()
    df = df.copy()

    df["return"] = df["Close"].pct_change()
    df["cum_return"] = (1 + df["return"]).cumprod()

    # cum return ^ 1/n
    n = len(df)/252 # 252 is trading days in a year
    # if data in hours, divide n further by 8 (8 hours in trading day)

    CAGR = (df["cum_return"][-1])**(1/n) - 1

    return CAGR

def volatility(DF):
    df = DF.copy()

    df["return"] = df["Close"].pct_change()

    # volatility is just st dev of returns
    # annualised volatility is volatility multiplied by sqrt(252)
    vol = df["return"].std() * np.sqrt(252)

    return vol

In [18]:
def sharpe(DF, rf):
    df = DF.copy()

    return (cagr(df) - rf)/volatility(df)

def sortino(DF, rf):
    df = DF.copy()

    df["return"] = df["Close"].pct_change()

    neg_return = np.where(df["return"]>0, 0, df["return"])
    neg_vol = pd.Series(neg_return[neg_return != 0]).std() * np.sqrt(252)

    return (cagr(df) - rf)/neg_vol

for ticker in ohlcv_data:
    print("Sharpe for {} = {}".format(ticker, sharpe(ohlcv_data[ticker], 0.03)))
    print("Sortino for {} = {}".format(ticker, sortino(ohlcv_data[ticker], 0.03)))





Sharpe for AMZN = -0.049588309032027905
Sortino for AMZN = nan
Sharpe for MSFT = 0.24915852277002407
Sortino for MSFT = nan
Sharpe for GOOG = 2.220567788815151
Sortino for GOOG = nan


/var/folders/d5/1j3xxm3j58j4xgkds2ls2zs80000gn/T/ipykernel_11982/169489359.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  CAGR = (df["cum_return"][-1])**(1/n) - 1
/var/folders/d5/1j3xxm3j58j4xgkds2ls2zs80000gn/T/ipykernel_11982/169489359.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  CAGR = (df["cum_return"][-1])**(1/n) - 1
/var/folders/d5/1j3xxm3j58j4xgkds2ls2zs80000gn/T/ipykernel_11982/169489359.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by po